# Perenual: 그외 과일·꽃 5종
왼쪽 열쇠(Secrets)에 `PERENUAL_API_KEY`가 있어야 해요. 결과 `perenual_fruitflower.json`(키는 REDACTED 처리)을 보내주세요.

In [ ]:
# -*- coding: utf-8 -*-
"""
Perenual API로 그외 과일·꽃 5종 정보 받기 (서연 실행용)

실행:  코랩 셀에 이 파일 내용을 전부 붙여넣고 실행 (또는 python perenual_fruitflower.py)
- 실행하면 API 키를 물어봅니다. 키는 파일에 저장되지 않습니다.
- 요청 횟수: 검색 5번 + 상세 5번 = 약 10번 (무료 하루 100번)
결과 (코랩은 왼쪽 폴더 아이콘 /content, PC는 실행한 폴더):
- perenual_fruitflower.json : 받은 원본 전체
- perenual_fruitflower.csv  : 한눈에 보는 요약
"""
import csv
import json
import urllib.parse
import urllib.request
from getpass import getpass
from pathlib import Path

HERBS = {  # 한글 이름: 검색어 (과일·꽃 5종)
    "제라늄": "geranium",
    "칼랑코에": "kalanchoe",
    "수국": "hydrangea",
    "레몬": "lemon",
    "금귤": "kumquat",
}
BASE = "https://perenual.com/api/v2"
OUT = Path.cwd()  # 코랩이면 /content


def get(url):
    with urllib.request.urlopen(url, timeout=30) as r:
        return json.loads(r.read().decode("utf-8"))


def main():
    try:  # 구글 코랩: 왼쪽 열쇠(Secrets)에 PERENUAL_API_KEY 저장
        from google.colab import userdata
        key = userdata.get("PERENUAL_API_KEY").strip()
    except Exception:
        key = getpass("Perenual API 키를 붙여넣고 Enter (화면에 안 보여요): ").strip()
    results = {}
    rows = []
    for ko, q in HERBS.items():
        print(f"\n[{ko}] '{q}' 검색 중...")
        lst = get(f"{BASE}/species-list?key={key}&q={urllib.parse.quote(q)}")
        cands = lst.get("data", [])
        if not cands:
            print("  검색 결과 없음")
            rows.append({"식물": ko, "검색어": q, "상태": "검색 결과 없음"})
            continue
        for c in cands[:5]:
            print(f"  후보 id={c.get('id')}  {c.get('common_name')}  ({', '.join(c.get('scientific_name') or [])})")
        free = [c for c in cands if c.get("id") and c["id"] <= 3000]
        first = free[0] if free else cands[0]  # 무료 범위(1~3000) 후보를 먼저 고름
        pid = first.get("id")
        if pid is None or pid > 3000:
            print(f"  첫 후보 id={pid} → 무료 범위(1~3000) 밖이라 상세 정보를 못 받아요")
            rows.append({"식물": ko, "검색어": q, "id": pid, "상태": "무료 범위 밖"})
            results[ko] = {"search": cands[:5]}
            continue
        d = get(f"{BASE}/species/details/{pid}?key={key}")
        results[ko] = {"search": cands[:5], "details": d}
        hard = d.get("hardiness") or {}
        rows.append({
            "식물": ko, "검색어": q, "id": pid, "상태": "받음",
            "영문명": d.get("common_name"),
            "학명": ", ".join(d.get("scientific_name") or []),
            "생활형(cycle)": d.get("cycle"),
            "물주기(watering)": d.get("watering"),
            "햇빛(sunlight)": ", ".join(d.get("sunlight") or []) if isinstance(d.get("sunlight"), list) else d.get("sunlight"),
            "내한성(hardiness)": f"{hard.get('min')}~{hard.get('max')}" if hard else "",
            "번식(propagation)": ", ".join(d.get("propagation") or []) if isinstance(d.get("propagation"), list) else d.get("propagation"),
            "병해충(pest)": ", ".join(d.get("pest_susceptibility") or []) if isinstance(d.get("pest_susceptibility"), list) else d.get("pest_susceptibility"),
            "실내(indoor)": d.get("indoor"),
            "관리난이도(care_level)": d.get("care_level"),
            "설명(description)": (d.get("description") or "")[:300],
        })
        print(f"  상세 받음: {d.get('common_name')} / 물주기 {d.get('watering')} / 햇빛 {d.get('sunlight')}")

    text = json.dumps(results, ensure_ascii=False, indent=2).replace(key, "REDACTED")  # API 키가 파일에 남지 않게
    (OUT / "perenual_fruitflower.json").write_text(text, encoding="utf-8")
    cols = []
    for r in rows:
        for k in r:
            if k not in cols:
                cols.append(k)
    with open(OUT / "perenual_fruitflower.csv", "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        w.writerows(rows)
    print("\n완료: perenual_fruitflower.json, perenual_fruitflower.csv 저장")
    print("첫 후보가 원하는 식물이 아니면(예: 다른 바질 품종) 알려주세요. id를 골라 다시 받을 수 있어요.")


if __name__ == "__main__":
    main()
